# Краулинг записей IAVD

Весь код был запущен в конце апреля 2026 г.

In [ ]:
from time import sleep
from tqdm import tqdm

import requests
from bs4 import BeautifulSoup

import pandas as pd

In [ ]:
has_audio = []
all = (
    pd.DataFrame(
        columns=['entry', 'link']
    )
    .set_index('entry')
)

for i in tqdm(range(1, 77)):
    resp1 = requests.get(f'https://itelmen.fas.harvard.edu/dictionary/ru/?page={i}')
    soup1 = BeautifulSoup(resp1.text, 'html.parser')

    for item in soup1.find_all('div', attrs={'class': 'dictionary-list-item'}):
        link = item.find('a', attrs={'class': 'dictionary-list-item-word'})
        all.loc[link.get_text()] = link.get('href')
        if item.find('div', attrs={'class': 'dictionary-list-item-video has'}) != None:
            has_audio.append(item.find('a', attrs={'class': 'dictionary-list-item-word'}).get('href'))

In [ ]:
#all.to_csv('all_iavd.csv')

In [ ]:
lexical_entries_to_recordings2 = (
    pd.DataFrame(
        columns=[
            'name_of_recording', 
            'lexical_entry',
            'link_to_entry',
            'link_to_recording'
        ]
    )
    .set_index('name_of_recording')
)

for item in tqdm(has_audio):
    item_link = 'https://itelmen.fas.harvard.edu' + item
    resp = requests.get(item_link)
    soup = BeautifulSoup(resp.text, 'html.parser')

    for but in soup.find_all('button'):
        text = but.get_text(strip=True)[1:]
        if 'ʔ' in text:
            file_name = f'FULL_itelmen_recordings/{text}.mp3'
            try:
                audio_link = 'https://itelmen.fas.harvard.edu' + but.find('audio')['src']
            except TypeError:
                print(f'TypeError on {item_link}. Continuing')
            audio_resp = requests.get(audio_link)

            with open(file_name.replace('?', 'QUESTIONMARK'), 'wb') as file:
                file.write(audio_resp.content)

            lexical_entries_to_recordings2.loc[file_name] = [soup.find('h4').get_text(), item_link, audio_link]

        sleep(1)

In [ ]:
#lexical_entries_to_recordings2.to_csv('lexical_entries_to_recordings2(FULL).csv')

In [ ]:
#lexical_entries_to_recordings2 = pd.read_csv('lexical_entries_to_recordings2(FULL).csv', index_col='name_of_recording')